# LAB 03 — Word Representations and Embeddings
**Môn:** Xử lý ngôn ngữ tự nhiên và ứng dụng - **Dữ liệu:** C4 (30K documents, file `c4-train_00000-of-01024-30K_json.gz`)


In [ ]:
!pip install -q gensim scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 43.0 MB/s eta 0:00:00


## 0. Cấu hình, nạp dữ liệu C4 và tiền xử lý

In [ ]:
# ===== 0. SETUP =====
import os, re, gzip, json, time, random, collections, math
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib
matplotlib.use("Agg") if os.environ.get("LAB_HEADLESS") else None
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from gensim.test.utils import datapath

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_PATH = os.environ.get("LAB_DATA", "c4-train_00000-of-01024-30K_json.gz")
N_DOCS    = int(os.environ.get("LAB_NDOCS", 30000))   # số document dùng để train (tối đa 30000)
EPOCHS    = int(os.environ.get("LAB_EPOCHS", 10))
WORKERS   = 1
OUT_DIR   = os.environ.get("LAB_OUT", ".")
R = {}
ROWS = []

if not os.path.exists(DATA_PATH):
    from google.colab import files
    up = files.upload()
    DATA_PATH = list(up.keys())[0]
print("Data:", DATA_PATH, "| N_DOCS =", N_DOCS, "| EPOCHS =", EPOCHS)

Saving c4-train.00000-of-01024-30K.json.gz to c4-train.00000-of-01024-30K.json.gz
Data: c4-train.00000-of-01024-30K.json.gz | N_DOCS = 30000 | EPOCHS = 10


In [ ]:
# ===== 0. LOAD + TOKENIZE =====
docs = []
with gzip.open(DATA_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line)["text"])
print("Tổng số documents trong file:", len(docs))

TOKEN_RE = re.compile(r"[a-z]+")
SPLIT_RE = re.compile(r"[\n.!?;]+")

sentences = []
for d in docs[:N_DOCS]:
    for s in SPLIT_RE.split(d.lower()):
        w = TOKEN_RE.findall(s)
        if len(w) >= 3:
            sentences.append(w)

freq = collections.Counter(w for s in sentences for w in s)
R["corpus"] = dict(docs_in_file=len(docs), docs_used=N_DOCS, sentences=len(sentences),
                   tokens=int(sum(freq.values())), types=len(freq),
                   types_min_count5=int(sum(1 for c in freq.values() if c >= 5)))
print(R["corpus"])

Tổng số documents trong file: 30000
{'docs_in_file': 30000, 'docs_used': 30000, 'sentences': 636689, 'tokens': 10803024, 'types': 166977, 'types_min_count5': 49709}


## CORE IMPLEMENTATION (Mục 11) — tự cài đặt, không dùng Word2Vec
`build_vocabulary()` → `build_cooccurrence_matrix()` → `cosine_similarity()` → `most_similar()`

In [ ]:
# ===== MỤC 11: CORE IMPLEMENTATION =====
def build_vocabulary(sentences, min_count=5, max_size=None, exclude=None, allowed=None):
    """Trả về (vocab: word->id, itos: list, counts: Counter). Sắp xếp theo tần suất giảm dần."""
    counts = collections.Counter(w for s in sentences for w in s)
    words = [w for w, c in counts.most_common() if c >= min_count]
    if allowed is not None: words = [w for w in words if w in allowed]
    if exclude is not None: words = [w for w in words if w not in exclude]
    if max_size is not None: words = words[:max_size]
    return {w: i for i, w in enumerate(words)}, words, counts

def build_cooccurrence_matrix(sentences, vocab, window):
    """Ma trận X (|V| x |V|): X[i,j] = số lần từ j nằm trong cửa sổ +-window quanh từ i (trong cùng câu).
    Từ OOV vẫn chiếm một vị trí (để khoảng cách giữ nguyên) nhưng không được đếm."""
    V = len(vocab)
    ids = np.fromiter((vocab.get(w, -1) for s in sentences for w in s), dtype=np.int32)
    sid = np.fromiter((i for i, s in enumerate(sentences) for _ in s), dtype=np.int32)
    X = sp.csr_matrix((V, V), dtype=np.float32)
    for d in range(1, window + 1):                       # mỗi khoảng cách d = 1..window
        a, b = ids[:-d], ids[d:]
        ok = (sid[:-d] == sid[d:]) & (a >= 0) & (b >= 0)
        a, b = a[ok], b[ok]
        r, c = np.concatenate([a, b]), np.concatenate([b, a])   # đối xứng
        X = X + sp.coo_matrix((np.ones(len(r), np.float32), (r, c)), shape=(V, V)).tocsr()
    return X

def cosine_similarity(u, v):
    """cos(u,v) = u.v / (|u||v|). Nhận vector dense hoặc hàng sparse."""
    u = u.toarray().ravel() if sp.issparse(u) else np.asarray(u, float)
    v = v.toarray().ravel() if sp.issparse(v) else np.asarray(v, float)
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return 0.0 if nu == 0 or nv == 0 else float(u @ v / (nu * nv))

def most_similar(word, matrix, vocabulary, top_k=5, norms=None):
    """Top-k từ gần `word` nhất theo cosine (loại chính nó)."""
    itos = sorted(vocabulary, key=vocabulary.get)
    i = vocabulary[word]
    if sp.issparse(matrix):
        dots = matrix.dot(matrix[i].T).toarray().ravel()
        if norms is None: norms = np.sqrt(np.asarray(matrix.multiply(matrix).sum(1)).ravel())
    else:
        dots = matrix @ matrix[i]
        if norms is None: norms = np.linalg.norm(matrix, axis=1)
    sims = dots / (norms * norms[i] + 1e-12)
    sims[i] = -np.inf
    top = np.argsort(-sims)[:top_k]
    return [(itos[j], float(sims[j])) for j in top]

def ppmi(X, alpha=0.75):
    """(Bổ sung) Positive PMI với context-distribution smoothing — giảm ảnh hưởng của từ rất phổ biến."""
    X = X.tocoo()
    rs = np.asarray(X.sum(1)).ravel(); cs = np.asarray(X.sum(0)).ravel() ** alpha
    tot = X.sum(); cs_tot = cs.sum()
    pmi = np.log((X.data / tot) / ((rs[X.row] / tot) * (cs[X.col] / cs_tot) + 1e-12) + 1e-12)
    pmi = np.maximum(pmi, 0)
    return sp.coo_matrix((pmi.astype(np.float32), (X.row, X.col)), shape=X.shape).tocsr()

## EXPERIMENT 1 (Mục 10) — Word-context matrix: corpus nhỏ trước, rồi tới C4

In [ ]:
# ===== EXPERIMENT 1a: corpus nhỏ (corpus ở Mục 5) với window 1, 2, 5 =====
toy_corpus = ["the cat eats fish", "the cat likes milk", "the dog eats meat", "the dog likes fish"]
toy_sents = [l.split() for l in toy_corpus]
toy_vocab, toy_itos, _ = build_vocabulary(toy_sents, min_count=1, exclude={"the"})
for w in (1, 2, 5):
    Xt = build_cooccurrence_matrix(toy_sents, toy_vocab, w)
    print(f"window={w}: matrix {Xt.shape}, non-zero = {Xt.nnz}")
    if w == 1: print(pd.DataFrame(Xt.toarray().astype(int), index=toy_itos, columns=toy_itos))
print("cos(cat,dog) window=1:", cosine_similarity(build_cooccurrence_matrix(toy_sents, toy_vocab, 1)[toy_vocab['cat']],
                                                  build_cooccurrence_matrix(toy_sents, toy_vocab, 1)[toy_vocab['dog']]))

window=1: matrix (7, 7), non-zero = 16
       cat  eats  fish  likes  dog  milk  meat
cat      0     1     0      1    0     0     0
eats     1     0     1      0    1     0     1
fish     0     1     0      1    0     0     0
likes    1     0     1      0    1     1     0
dog      0     1     0      1    0     0     0
milk     0     0     0      1    0     0     0
meat     0     1     0      0    0     0     0
window=2: matrix (7, 7), non-zero = 24
window=5: matrix (7, 7), non-zero = 24
cos(cat,dog) window=1: 1.0000001192092896


In [ ]:
# ===== EXPERIMENT 1b: C4, window = 1, 2, 5 =====
vocab_c, itos_c, counts_c = build_vocabulary(sentences, min_count=5, max_size=20000)
print("Vocabulary (min_count=5, top 20000):", len(vocab_c))
PAIRS = [("doctor","physician"), ("doctor","hospital"), ("cat","dog"), ("car","automobile"), ("doctor","banana")]
exp1 = []
X_by_w, P_by_w = {}, {}
for w in (1, 2, 5):
    t0 = time.time()
    X = build_cooccurrence_matrix(sentences, vocab_c, w)
    P = ppmi(X)
    nbytes = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
    row = dict(window=w, vocab=len(vocab_c), shape=f"{X.shape[0]}x{X.shape[1]}", total_cells=X.shape[0]*X.shape[1],
               nnz=int(X.nnz), density_pct=100 * X.nnz / (X.shape[0] * X.shape[1]),
               sparse_MB=nbytes / 1e6, dense_GB=X.shape[0]*X.shape[1]*4 / 1e9, build_s=time.time() - t0)
    for a, b in PAIRS:
        if a in vocab_c and b in vocab_c:
            row[f"raw_{a}-{b}"]  = cosine_similarity(X[vocab_c[a]], X[vocab_c[b]])
            row[f"ppmi_{a}-{b}"] = cosine_similarity(P[vocab_c[a]], P[vocab_c[b]])
    row["raw_doctor_top5"]  = most_similar("doctor", X, vocab_c, 5)
    row["ppmi_doctor_top5"] = most_similar("doctor", P, vocab_c, 5)
    exp1.append(row); X_by_w[w], P_by_w[w] = X, P if w == 5 else None
    for k, v in row.items():
        ROWS.append(dict(experiment="exp1_cooc", item=k, setting=f"window={w}", value=str(v)))
    if w != 5: del X, P
R["exp1"] = exp1
print(pd.DataFrame([{k: v for k, v in r.items() if "top5" not in k} for r in exp1]).T)
for r in exp1:
    print(f"window={r['window']}  raw  top5(doctor):", [(a, round(s, 2)) for a, s in r["raw_doctor_top5"]])
    print(f"window={r['window']}  PPMI top5(doctor):", [(a, round(s, 2)) for a, s in r["ppmi_doctor_top5"]])
P5 = P_by_w[5]; del X_by_w, P_by_w        # giữ PPMI window=5 cho error analysis
import gc; gc.collect()

Vocabulary (min_count=5, top 20000): 20000
                                 0            1            2
window                           1            2            5
vocab                        20000        20000        20000
shape                  20000x20000  20000x20000  20000x20000
total_cells              400000000    400000000    400000000
nnz                        3481847      6645934     13415803
density_pct               0.870462     1.661483     3.353951
sparse_MB                 27.93478    53.247476   107.406428
dense_GB                       1.6          1.6          1.6
build_s                   9.699049     6.558003    10.780589
raw_doctor-physician       0.92503     0.933666     0.953542
ppmi_doctor-physician     0.159101     0.180477     0.210659
raw_doctor-hospital       0.695731     0.774305      0.84986
ppmi_doctor-hospital      0.099842     0.095202     0.152312
raw_cat-dog               0.892398     0.927124     0.956606
ppmi_cat-dog              0.176178     0.1

0

## Mục 12 → Mục 13: từ co-occurrence tới dense embedding
Ma trận 100,000 x 100,000 có 10^10 ô (≈ 40 GB nếu lưu dense float32) → cần biểu diễn dense, chiều thấp (Word2Vec).

## EXPERIMENT 2 (Mục 17) — Word2Vec: ghi rõ mọi tham số

In [ ]:
# ===== EXPERIMENT 2: Word2Vec (skip-gram + negative sampling) =====
os.makedirs(os.path.join(OUT_DIR, "models"), exist_ok=True)
def train_w2v(vector_size=100, window=5, sg=1, min_count=5, negative=5, epochs=EPOCHS, name=None):
    path = os.path.join(OUT_DIR, "models", f"{name}.model") if name else None
    if path and os.path.exists(path) and os.path.exists(path + ".time"):      # đã train rồi -> nạp lại (resume)
        return Word2Vec.load(path), float(open(path + ".time").read())
    t0 = time.time()
    m = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=min_count, sg=sg,
                 negative=negative, epochs=epochs, workers=WORKERS, seed=SEED)
    secs = time.time() - t0
    if path:
        m.save(path); open(path + ".time", "w").write(str(secs))
    return m, secs

CONFIG = dict(corpus=f"C4 subset: {N_DOCS} docs, {len(sentences)} sentences, {R['corpus']['tokens']} tokens",
              vector_size=100, window=5, min_count=5, epochs=EPOCHS, sg=1, negative=5,
              objective="Skip-gram with negative sampling (SGNS): dự đoán context từ target, 5 negative samples")
m_base, t_base = train_w2v(100, 5, name="w5_d100")
CONFIG["vocabulary"] = len(m_base.wv); CONFIG["train_seconds"] = t_base
R["exp2_config"] = CONFIG
for k, v in CONFIG.items(): print(f"{k:14s}: {v}")

# (bổ sung) CBOW cùng cấu hình để so sánh với Skip-gram
m_cbow, t_cbow = train_w2v(100, 5, sg=0, name="cbow_w5_d100")
print("CBOW train seconds:", round(t_cbow, 1))

corpus        : C4 subset: 30000 docs, 636689 sentences, 10803024 tokens
vector_size   : 100
window        : 5
min_count     : 5
epochs        : 10
sg            : 1
negative      : 5
objective     : Skip-gram with negative sampling (SGNS): dự đoán context từ target, 5 negative samples
vocabulary    : 49709
train_seconds : 625.6987714767456
CBOW train seconds: 196.6


## Mục 18 — Inspect embeddings: Top-5 neighbours

In [ ]:
# ===== MỤC 18: Top-5 most_similar =====
INSPECT = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
top5 = {}
for w in INSPECT:
    top5[w] = {"skipgram": [(a, round(s, 3)) for a, s in m_base.wv.most_similar(w, topn=5)],
               "cbow":     [(a, round(s, 3)) for a, s in m_cbow.wv.most_similar(w, topn=5)],
               "top10_sg": [(a, round(s, 3)) for a, s in m_base.wv.most_similar(w, topn=10)]}
    print(f"{w:9s} SG  :", top5[w]["skipgram"]); print(f"{'':9s} CBOW:", top5[w]["cbow"])
R["inspect_top5"] = top5

# --- Evidence từ corpus: các context word có PPMI cao nhất của mỗi từ (window=5) + câu ví dụ ---
CTX_OK = np.array([counts_c[w] >= 50 for w in itos_c], dtype=np.float32)
def top_contexts(word, n=8):
    row = P5[vocab_c[word]].toarray().ravel() * CTX_OK      # chỉ xét context word có tần suất >= 50
    return [itos_c[j] for j in np.argsort(-row)[:n]]
def sentences_with(*words, limit=2):
    out = []
    for s in sentences:
        if all(w in s for w in words):
            out.append(" ".join(s[:30]))
            if len(out) >= limit: break
    return out
def n_sent_with(*words):
    return sum(1 for s in sentences if all(w in s for w in words))

evidence = {}
for w in INSPECT:
    evidence[w] = dict(freq=int(freq[w]), top_contexts=top_contexts(w),
                       neighbours=[a for a, _ in top5[w]["skipgram"]])
    print(w, "| freq =", freq[w], "| top PPMI contexts:", evidence[w]["top_contexts"])
for a, b in [("doctor","hospital"), ("doctor","patient"), ("hospital","patient"), ("hospital","disease")]:
    print(f"sentences chứa cả '{a}' và '{b}':", n_sent_with(a, b))
    evidence[f"{a}+{b}"] = dict(n_sentences=n_sent_with(a, b), examples=sentences_with(a, b))
for s in sentences_with("doctor", "hospital", limit=2): print("  ví dụ:", s)
R["inspect_evidence"] = evidence

doctor    SG  : [('dentist', 0.788), ('ophthalmologist', 0.766), ('physician', 0.763), ('obstetrician', 0.759), ('veterinarian', 0.755)]
          CBOW: [('dentist', 0.812), ('physician', 0.786), ('veterinarian', 0.753), ('surgeon', 0.697), ('therapist', 0.687)]
hospital  SG  : [('upmc', 0.777), ('samitivej', 0.706), ('icu', 0.703), ('hospice', 0.691), ('medical', 0.682)]
          CBOW: [('clinic', 0.769), ('rehab', 0.665), ('chiropractor', 0.658), ('clinics', 0.653), ('hospice', 0.651)]
patient   SG  : [('patients', 0.722), ('antenatal', 0.672), ('individualised', 0.665), ('retinal', 0.661), ('neonatal', 0.655)]
          CBOW: [('patients', 0.665), ('physician', 0.665), ('child', 0.623), ('diagnosis', 0.621), ('doctor', 0.597)]
disease   SG  : [('diseases', 0.847), ('autoimmune', 0.788), ('chronic', 0.787), ('infection', 0.785), ('diabetes', 0.774)]
          CBOW: [('diseases', 0.787), ('infection', 0.776), ('chronic', 0.761), ('hypertension', 0.757), ('lung', 0.749)]
computer  SG 

## EXPERIMENT 3 (Mục 19) — Ảnh hưởng của context window

In [ ]:
# ===== EXPERIMENT 3: window = 2, 5, 10 (dim=100) =====
WS_FILE = datapath("wordsim353.tsv")
models_w = {5: (m_base, t_base)}
for w in (2, 10):
    models_w[w] = train_w2v(100, w, name=f"w{w}_d100")
PAIRS3 = [("doctor","physician"), ("doctor","hospital"), ("cat","dog"), ("car","automobile"), ("doctor","banana")]
exp3 = {}
for w in (2, 5, 10):
    m, secs = models_w[w]
    sims = {f"{a}-{b}": round(float(m.wv.similarity(a, b)), 3) for a, b in PAIRS3 if a in m.wv and b in m.wv}
    (_, _), (sp_r, _), oov = m.wv.evaluate_word_pairs(WS_FILE)
    exp3[w] = dict(pairs=sims, wordsim_spearman=round(float(sp_r), 4), wordsim_oov_pct=round(float(oov), 1),
                   train_s=round(secs, 1), doctor_top5=[a for a, _ in m.wv.most_similar("doctor", topn=5)])
    for k, v in sims.items(): ROWS.append(dict(experiment="exp3_window", item=k, setting=f"window={w}", value=v))
    ROWS.append(dict(experiment="exp3_window", item="wordsim353_spearman", setting=f"window={w}", value=exp3[w]["wordsim_spearman"]))
R["exp3"] = exp3
print(pd.DataFrame({f"window={w}": exp3[w]["pairs"] for w in exp3}))
for w in exp3: print(w, "WordSim353 Spearman:", exp3[w]["wordsim_spearman"], "| OOV %:", exp3[w]["wordsim_oov_pct"],
                     "| doctor top5:", exp3[w]["doctor_top5"], "| train s:", exp3[w]["train_s"])

                  window=2  window=5  window=10
doctor-physician     0.779     0.763      0.749
doctor-hospital      0.530     0.559      0.574
cat-dog              0.696     0.727      0.755
car-automobile       0.720     0.734      0.728
doctor-banana        0.034     0.018      0.069
2 WordSim353 Spearman: 0.5815 | OOV %: 2.8 | doctor top5: ['physician', 'dentist', 'veterinarian', 'obstetrician', 'midwife'] | train s: 325.2
5 WordSim353 Spearman: 0.6217 | OOV %: 2.8 | doctor top5: ['dentist', 'ophthalmologist', 'physician', 'obstetrician', 'veterinarian'] | train s: 625.7
10 WordSim353 Spearman: 0.6345 | OOV %: 2.8 | doctor top5: ['obstetrician', 'alendronate', 'endocrinologist', 'physician', 'dentist'] | train s: 978.2


## EXPERIMENT 4 (Mục 20) — Ảnh hưởng của embedding dimension (50 / 100 / 300)
Đánh giá: thời gian train, kích thước model, similarity, analogy (Google analogy set), và một downstream task đơn giản
(phân loại từ vào 5 nhóm chủ đề bằng Logistic Regression + 5-fold CV).

In [ ]:
# ===== EXPERIMENT 4: dimension 50 / 100 / 300 (window=5) =====
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
CATS = {
 "medical":    "doctor nurse hospital patient disease surgery medicine health clinic physician cancer treatment".split(),
 "sports":     "football basketball baseball soccer tennis hockey coach team season player league stadium".split(),
 "food":       "banana apple bread cheese pizza chicken rice pasta chocolate soup salad butter".split(),
 "technology": "computer software internet laptop website smartphone data network server digital programming email".split(),
 "vehicles":   "car truck bus train bicycle motorcycle engine driver road airplane taxi vehicle".split(),
}
def downstream_acc(m):
    Xw, yw = [], []
    for ci, (c, ws) in enumerate(CATS.items()):
        for w in ws:
            if w in m.wv: Xw.append(m.wv[w] / np.linalg.norm(m.wv[w])); yw.append(ci)
    clf = LogisticRegression(max_iter=2000)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    return float(cross_val_score(clf, np.array(Xw), np.array(yw), cv=cv).mean()), len(yw)

models_d = {100: (m_base, t_base)}
for d in (50, 300):
    models_d[d] = train_w2v(d, 5, name=f"w5_d{d}")
exp4 = {}
for d in (50, 100, 300):
    m, secs = models_d[d]
    (_, _), (sp_r, _), oov = m.wv.evaluate_word_pairs(WS_FILE)
    ana_acc, sections = m.wv.evaluate_word_analogies(datapath("questions-words.txt"), case_insensitive=True, restrict_vocab=20000)
    n_ok = sum(len(s["correct"]) for s in sections[:-1]); n_bad = sum(len(s["incorrect"]) for s in sections[:-1])
    acc_cls, n_words = downstream_acc(m)
    path = os.path.join(OUT_DIR, "models", f"w5_d{d}.model")
    exp4[d] = dict(train_s=round(secs, 1), vectors_MB=round(m.wv.vectors.nbytes / 1e6, 2),
                   file_MB=round(os.path.getsize(path) / 1e6, 2) if os.path.exists(path) else None,
                   pairs={f"{a}-{b}": round(float(m.wv.similarity(a, b)), 3) for a, b in PAIRS3 if a in m.wv and b in m.wv},
                   wordsim_spearman=round(float(sp_r), 4), analogy_acc=round(float(ana_acc), 4),
                   analogy_answered=n_ok + n_bad, downstream_acc=round(acc_cls, 3), downstream_n=n_words)
    for k in ("train_s", "vectors_MB", "wordsim_spearman", "analogy_acc", "downstream_acc"):
        ROWS.append(dict(experiment="exp4_dim", item=k, setting=f"dim={d}", value=exp4[d][k]))
R["exp4"] = exp4
print(pd.DataFrame(exp4).T.drop(columns=["pairs"]))
print(pd.DataFrame({f"dim={d}": exp4[d]["pairs"] for d in exp4}))

    train_s vectors_MB file_MB wordsim_spearman analogy_acc analogy_answered  \
50    637.2       9.94   21.39           0.5929      0.2792            10494   
100   625.7      19.88   41.28           0.6217      0.3211            10494   
300   941.9      59.65    1.51           0.6026      0.3106            10494   

    downstream_acc downstream_n  
50             1.0           60  
100            1.0           60  
300            1.0           60  
                  dim=50  dim=100  dim=300
doctor-physician   0.847    0.763    0.602
doctor-hospital    0.642    0.559    0.376
cat-dog            0.845    0.727    0.468
car-automobile     0.807    0.734    0.549
doctor-banana      0.007    0.018   -0.006


## EVALUATION — Word similarity (Mục 21): xếp hạng + so sánh trực giác + định lượng

In [ ]:
# ===== MỤC 21: Word similarity =====
from scipy.stats import spearmanr
EVAL_PAIRS = [("doctor","physician", 9.0), ("car","automobile", 9.0), ("king","queen", 8.0),
              ("cat","dog", 7.0), ("computer","banana", 0.5)]   # cột 3: điểm trực giác (tự gán, thang 0-10)
rows = []
for a, b, gold in EVAL_PAIRS:
    rows.append(dict(pair=f"{a}-{b}", intuition=gold, cos=float(m_base.wv.similarity(a, b))))
ev = pd.DataFrame(rows)
ev["rank_model"] = ev["cos"].rank(ascending=False).astype(int)
ev["rank_intuition"] = ev["intuition"].rank(ascending=False, method="min").astype(int)
rho5 = spearmanr(ev["intuition"], ev["cos"])
print(ev.sort_values("cos", ascending=False)); print("Spearman (5 cặp, trực giác):", rho5.correlation)
(_, _), (sp_r, sp_p), oov = m_base.wv.evaluate_word_pairs(WS_FILE)
print(f"WordSim-353: Spearman = {sp_r:.4f} (p={sp_p:.2g}), OOV = {oov:.1f}%")
R["eval_similarity"] = dict(table=ev.to_dict("records"), spearman5=float(rho5.correlation),
                            wordsim353_spearman=float(sp_r), wordsim353_p=float(sp_p), wordsim353_oov=float(oov))
for r in ev.to_dict("records"): ROWS.append(dict(experiment="eval_similarity", item=r["pair"], setting="w5_d100", value=round(r["cos"], 4)))

               pair  intuition       cos  rank_model  rank_intuition
2        king-queen        8.0  0.767658           1               3
0  doctor-physician        9.0  0.762869           2               1
1    car-automobile        9.0  0.733661           3               1
3           cat-dog        7.0  0.727469           4               4
4   computer-banana        0.5  0.055245           5               5
Spearman (5 cặp, trực giác): 0.6668859288553503
WordSim-353: Spearman = 0.6217 (p=4.6e-38), OOV = 2.8%


## EVALUATION — Word analogy (Mục 22) và Bài 23

In [ ]:
# ===== MỤC 22: Word analogy =====
ana = {}
tests = [(["king", "woman"], ["man"], "queen"), (["paris", "italy"], ["france"], "rome"),
         (["brother", "woman"], ["man"], "sister"), (["walking", "swam"], ["walked"], "swimming")]
for pos, neg, target in tests:
    if all(w in m_base.wv for w in pos + neg):
        res = m_base.wv.most_similar(positive=pos, negative=neg, topn=5)
        rank = next((i + 1 for i, (w, _) in enumerate(m_base.wv.most_similar(positive=pos, negative=neg, topn=1000)) if w == target), None)
        ana[f"{'+'.join(pos)}-{'-'.join(neg)}"] = dict(expected=target, top5=[(a, round(s, 3)) for a, s in res], rank_of_expected=rank)
        print(f"{' + '.join(pos)} - {' - '.join(neg)}  (kỳ vọng: {target}) → top5:", [(a, round(s, 2)) for a, s in res], "| rank:", rank)
(acc, secs_) = m_base.wv.evaluate_word_analogies(datapath("questions-words.txt"), case_insensitive=True, restrict_vocab=20000)
sec_tbl = [(s["section"], len(s["correct"]), len(s["correct"]) + len(s["incorrect"])) for s in secs_ if len(s["correct"]) + len(s["incorrect"]) > 0]
print("Google analogy accuracy (trên các câu có đủ từ):", round(acc, 4))
for s, c, n in sec_tbl: print(f"  {s[:34]:34s} {c}/{n}")
R["analogy"] = dict(examples=ana, google_acc=float(acc), sections=sec_tbl)

## APPLICATION (Mục 24) — Semantic search & clustering, so sánh với TF-IDF của LAB 01

In [ ]:
# ===== MỤC 24: Semantic search =====
from sklearn.feature_extraction.text import TfidfVectorizer
SEARCH_DOCS = [d for d in docs[:5000] if len(d.split()) >= 40]
print("Số documents để tìm kiếm:", len(SEARCH_DOCS))

# (a) Baseline LAB 01: TF-IDF + cosine
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2, token_pattern=r"[a-z]+")
T = tfidf.fit_transform(SEARCH_DOCS)

# (b) LAB 03: document vector = trung bình trọng số TF-IDF của word embeddings
vocab_w = m_base.wv.index_to_key
tf2 = TfidfVectorizer(vocabulary=vocab_w, stop_words=None, sublinear_tf=True, token_pattern=r"[a-z]+")
T2 = tf2.fit_transform(SEARCH_DOCS)
E = m_base.wv.vectors / np.linalg.norm(m_base.wv.vectors, axis=1, keepdims=True)
D = T2 @ E
D = D / (np.linalg.norm(D, axis=1, keepdims=True) + 1e-9)

def query_vec(q):
    ws = [w for w in TOKEN_RE.findall(q.lower()) if w in m_base.wv]
    v = np.mean([E[m_base.wv.key_to_index[w]] for w in ws], axis=0)
    return v / np.linalg.norm(v)
def search(q, k=5):
    s_emb = D @ query_vec(q)
    s_tf = (T @ tfidf.transform([q]).T).toarray().ravel()
    expand = {w: [a for a, _ in m_base.wv.most_similar(w, topn=6)] for w in TOKEN_RE.findall(q.lower()) if w in m_base.wv}
    pick = lambda s: [(int(i), float(s[i]), SEARCH_DOCS[i][:140].replace("\n", " ")) for i in np.argsort(-s)[:k]]
    return dict(query=q, expansion=expand, embedding=pick(s_emb), tfidf=pick(s_tf))

search_res = [search(q) for q in ["medical treatment", "football game", "computer software"]]
for r in search_res:
    print("\n=== QUERY:", r["query"]); print("Từ gần (query expansion):", r["expansion"])
    for name in ("embedding", "tfidf"):
        print(f"-- Top-5 {name}:")
        for i, s, t in r[name]: print(f"   [{i}] {s:.3f} | {t}")
R["semantic_search"] = search_res

# (c) Clustering các từ bằng embedding (KMeans k=5) + PCA 2D
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
cw, cl = [], []
for ci, (c, ws) in enumerate(CATS.items()):
    for w in ws:
        if w in m_base.wv: cw.append(w); cl.append(ci)
Xc = np.array([m_base.wv[w] / np.linalg.norm(m_base.wv[w]) for w in cw])
km = KMeans(5, n_init=10, random_state=SEED).fit(Xc)
purity = sum(collections.Counter(np.array(cl)[km.labels_ == k]).most_common(1)[0][1] for k in range(5)) / len(cl)
print("KMeans purity:", round(purity, 3))
clusters = {int(k): [cw[i] for i in range(len(cw)) if km.labels_[i] == k] for k in range(5)}
for k, v in clusters.items(): print(k, v)
R["clustering"] = dict(purity=float(purity), clusters=clusters)
Z = PCA(2, random_state=SEED).fit_transform(Xc)
plt.figure(figsize=(8, 6))
for ci, c in enumerate(CATS):
    idx = [i for i in range(len(cw)) if cl[i] == ci]
    plt.scatter(Z[idx, 0], Z[idx, 1], label=c)
    for i in idx: plt.annotate(cw[i], Z[i], fontsize=7)
plt.legend(); plt.title("PCA 2D của word embeddings (Word2Vec, dim=100)"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_pca.png"), dpi=150); plt.show()

## ERROR ANALYSIS (Mục 25) — 3 similarity đúng + 3 similarity sai/bất ngờ

In [ ]:
# ===== MỤC 25: Error analysis helper =====
def explain_pair(w1, w2):
    m = m_base
    nb = [a for a, _ in m.wv.most_similar(w1, topn=1000)]
    d = dict(pair=f"{w1}-{w2}", cos=round(float(m.wv.similarity(w1, w2)), 3), freq1=int(freq[w1]), freq2=int(freq[w2]),
             rank_of_w2_in_neighbours_of_w1=(nb.index(w2) + 1 if w2 in nb else None),
             sentences_with_both=n_sent_with(w1, w2), contexts1=top_contexts(w1, 6), contexts2=top_contexts(w2, 6),
             shared=sorted(set(top_contexts(w1, 30)) & set(top_contexts(w2, 30)))[:8], example=sentences_with(w1, w2, limit=1))
    return d
ERR_PAIRS_OK  = [("doctor","physician"), ("doctor","surgeon"), ("computer","laptop")]      # đúng / hợp lý
ERR_PAIRS_BAD = [("doctor","hospital"), ("football","replica"), ("banana","onions")]     # sai / bất ngờ (chọn theo quan sát ở Mục 18)
err = {"ok": [explain_pair(*p) for p in ERR_PAIRS_OK], "bad": [explain_pair(*p) for p in ERR_PAIRS_BAD]}
for g in ("ok", "bad"):
    for e in err[g]: print(g.upper(), json.dumps(e, ensure_ascii=False))
R["error_analysis"] = err

## POLYSEMY (Mục 26) — từ `bank`
Static Word2Vec chỉ có **một** vector cho `bank` dù có ít nhất hai nghĩa (ngân hàng / bờ sông).

In [ ]:
# ===== MỤC 26: Polysemy — bank =====
bank_nb = [(a, round(s, 3)) for a, s in m_base.wv.most_similar("bank", topn=15)]
print("Top-15 gần 'bank':", bank_nb)
MONEY = set("money account loan credit deposit financial interest card savings branch mortgage cash".split())
RIVER = set("river water shore stream lake fishing mud flood bridge trees fish".split())
n_money = n_river = n_total = 0; ex_money = ex_river = None
for s in sentences:
    if "bank" in s:
        n_total += 1; ss = set(s)
        if ss & MONEY:
            n_money += 1; ex_money = ex_money or " ".join(s[:25])
        if ss & RIVER:
            n_river += 1; ex_river = ex_river or " ".join(s[:25])
print(f"Câu chứa 'bank': {n_total} | có ngữ cảnh tiền: {n_money} | có ngữ cảnh sông/nước: {n_river}")
sims_bank = {w: round(float(m_base.wv.similarity("bank", w)), 3) for w in ["money", "loan", "account", "river", "water", "shore"] if w in m_base.wv}
print(sims_bank)
R["polysemy"] = dict(neighbours=bank_nb, n_total=n_total, n_money=n_money, n_river=n_river,
                     sims=sims_bank, ex_money=ex_money, ex_river=ex_river)

Top-15 gần 'bank': [('banks', 0.752), ('citibank', 0.733), ('anz', 0.728), ('credit', 0.724), ('banking', 0.702), ('barclays', 0.697), ('redstone', 0.696), ('debit', 0.692), ('hsbc', 0.69), ('lloyds', 0.684), ('cardholder', 0.676), ('custodian', 0.666), ('phoebus', 0.664), ('suntrust', 0.662), ('rbs', 0.656)]
Câu chứa 'bank': 1364 | có ngữ cảnh tiền: 472 | có ngữ cảnh sông/nước: 17
{'money': 0.455, 'loan': 0.623, 'account': 0.59, 'river': 0.351, 'water': 0.176, 'shore': 0.282}


## REFLECTION (Mục 27)
| Representation | Context-dependent? | Sparse/Dense | Một từ có nhiều vector? |
|---|---|---|---|
| TF-IDF | Không (đếm theo document, không có ngữ cảnh cục bộ) | Sparse | Không (1 chiều/từ; vector thuộc về document) |
| Co-occurrence | Một phần (dùng ngữ cảnh trong cửa sổ, nhưng một vector cố định/từ) | Sparse (có thể giảm chiều thành dense) | Không |
| Word2Vec | Không (static) | Dense | Không (1 vector/từ) |
| Contextual embedding | Có | Dense | Có (vector khác nhau theo câu) |

In [ ]:
# ===== LƯU KẾT QUẢ: results.json + results.csv =====
def to_py(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
with open(os.path.join(OUT_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(R, f, ensure_ascii=False, indent=1, default=to_py)
pd.DataFrame(ROWS).to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)
print("Đã lưu results.json và results.csv")